### PHASE 4: ETL - EXTRACT --> TRANSFORM --> LOAD

In [1]:
import pandas as pd
import numpy as np

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

### Configuration setup

In [2]:
RAW_DATA_PATH = "../data/raw/"
PROCESSED_DATA_PATH = "../data/processed/"

### Load all raw Data

In [3]:
customers = pd.read_csv(f"{RAW_DATA_PATH}olist_customers_dataset.csv")

orders = pd.read_csv(f"{RAW_DATA_PATH}olist_orders_dataset.csv")

order_items = pd.read_csv(f"{RAW_DATA_PATH}olist_order_items_dataset.csv")

payments = pd.read_csv(f"{RAW_DATA_PATH}olist_order_payments_dataset.csv")

reviews = pd.read_csv(f"{RAW_DATA_PATH}olist_order_reviews_dataset.csv")

products = pd.read_csv(f"{RAW_DATA_PATH}olist_products_dataset.csv")

sellers = pd.read_csv(f"{RAW_DATA_PATH}olist_sellers_dataset.csv")

geolocation = pd.read_csv(f"{RAW_DATA_PATH}olist_geolocation_dataset.csv")

category_translation = pd.read_csv(
    f"{RAW_DATA_PATH}product_category_name_translation.csv"
)

### Verify data loaded?

In [4]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation,
}

for name, df in datasets.items():
    print(f"{name:<25} {df.shape}")

Customers                 (99441, 5)
Orders                    (99441, 8)
Order Items               (112650, 7)
Payments                  (103886, 5)
Reviews                   (99224, 7)
Products                  (32951, 9)
Sellers                   (3095, 4)
Geolocation               (1000163, 5)
Category Translation      (71, 2)


### create working copies 

In [5]:
customers_etl = customers.copy()

orders_etl = orders.copy()

order_items_etl = order_items.copy()

payments_etl = payments.copy()

reviews_etl = reviews.copy()

products_etl = products.copy()

sellers_etl = sellers.copy()

geolocation_etl = geolocation.copy()

category_translation_etl = category_translation.copy()

### STEP 5: ETL CHECKLIST 

In [6]:
print("""
ETL Pipeline

1. Standardize Data Types
2. Handle Missing Values
3. Handle Data Quality Issues
4. Feature Engineering
5. Dataset Integration
6. Export Processed Data
""")


ETL Pipeline

1. Standardize Data Types
2. Handle Missing Values
3. Handle Data Quality Issues
4. Feature Engineering
5. Dataset Integration
6. Export Processed Data



### Step 6: check datatypes before standizing 


In [7]:
for name, df in {
    "Orders": orders_etl,
    "Payments": payments_etl,
    "Products": products_etl,
    "Reviews": reviews_etl
}.items():

    print("="*50)
    print(name)
    print(df.dtypes)

Orders
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object
Payments
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object
Products
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object
Reviews
review_id                    str
order_id                     str
review_score               int64
review_comment_title        

### Convert date columsn to date instead of string 

In [8]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders_etl[order_date_columns] = (
    orders_etl[order_date_columns]
    .apply(pd.to_datetime)
)

In [9]:
# Check the data types of the date columns after conversion
orders_etl[order_date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [10]:
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp",
]

reviews_etl[review_date_columns] = (
    reviews_etl[review_date_columns]
    .apply(pd.to_datetime)
)

In [11]:
#verify the data types of the review date columns after conversion
reviews_etl[review_date_columns].dtypes

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [12]:
delivery_time = (
    orders_etl["order_delivered_customer_date"]
    - orders_etl["order_purchase_timestamp"]
)
delivery_time.describe()

count                      96476
mean     12 days 13:24:31.879068
std       9 days 13:07:00.181125
min              0 days 12:48:07
25%       6 days 18:23:37.250000
50%             10 days 05:13:34
75%      15 days 17:17:16.250000
max            209 days 15:05:12
dtype: object

### create a summary instead of filling 

In [13]:
for name, df in {
    "Customers": customers_etl,
    "Orders": orders_etl,
    "Order Items": order_items_etl,
    "Payments": payments_etl,
    "Reviews": reviews_etl,
    "Products": products_etl,
    "Sellers": sellers_etl,
}.items():

    print("=" * 50)
    print(name)

    missing = (
        df.isna()
          .sum()
          .sort_values(ascending=False)
    )

    print(missing[missing > 0])

    print()

Customers
Series([], dtype: int64)

Orders
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

Order Items
Series([], dtype: int64)

Payments
Series([], dtype: int64)

Reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

Products
product_category_name         610
product_description_lenght    610
product_name_lenght           610
product_photos_qty            610
product_weight_g                2
product_height_cm               2
product_length_cm               2
product_width_cm                2
dtype: int64

Sellers
Series([], dtype: int64)



### create data quality flags 

Approval after Carrier


In [14]:
orders_etl["approval_after_carrier_flag"] = (
    orders_etl["order_approved_at"]
    >
    orders_etl["order_delivered_carrier_date"]
)

Late Delivery

In [15]:

orders_etl["late_delivery_flag"] = (
    orders_etl["order_delivered_customer_date"]
    >
    orders_etl["order_estimated_delivery_date"]
)

Delivery before Carrier

In [16]:

orders_etl["delivery_before_carrier_flag"] = (
    orders_etl["order_delivered_customer_date"]
    <
    orders_etl["order_delivered_carrier_date"]
)



Invalid Installments

In [17]:

payments_etl["invalid_installment_flag"] = (
    payments_etl["payment_installments"] <= 0
)


Verify Flags

In [18]:
print("Approval After Carrier")
print(orders_etl["approval_after_carrier_flag"] .value_counts())

print("\nLate Delivery")
print(orders_etl["late_delivery_flag"].value_counts())

print("\nDelivery Before Carrier")
print(orders_etl["delivery_before_carrier_flag"].value_counts())

print("\nInvalid Installments")
print(payments_etl["invalid_installment_flag"].value_counts())

Approval After Carrier
approval_after_carrier_flag
False    98082
True      1359
Name: count, dtype: int64

Late Delivery
late_delivery_flag
False    91614
True      7827
Name: count, dtype: int64

Delivery Before Carrier
delivery_before_carrier_flag
False    99418
True        23
Name: count, dtype: int64

Invalid Installments
invalid_installment_flag
False    103884
True          2
Name: count, dtype: int64


### we are filling only the column

In [19]:
reviews_etl["review_comment_title"] = (
    reviews_etl["review_comment_title"]
    .fillna("")
)

reviews_etl["review_comment_message"] = (
    reviews_etl["review_comment_message"]
    .fillna("")
)

# after filling check only count    
reviews_etl[
    ["review_comment_title", "review_comment_message"]
].isna().sum()

review_comment_title      0
review_comment_message    0
dtype: int64

In [20]:
for name, df in {
    "Orders": orders_etl,
    "Reviews": reviews_etl,
    "Products": products_etl,
}.items():

    print("=" * 40)
    print(name)
    print(df.isna().sum()[df.isna().sum() > 0])

Orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
Reviews
Series([], dtype: int64)
Products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [ ]:
fact_orders = orders_etl.copy()

fact_orders = pd.merge(
    fact_orders,
    customers,
    on="customer_id",
    how="left"
)


In [23]:
print(fact_orders.shape)

fact_orders.head()

(99441, 15)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_after_carrier_flag,late_delivery_flag,delivery_before_carrier_flag,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False,False,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False,False,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False,False,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False,False,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False,False,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


Step 12 – Merge Payments

Before merging, we need to answer one question:

Can one order have multiple payment records? 

In [24]:
#Step 1 — Check the relationship

payments.groupby("order_id").size().value_counts().sort_index()


1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

In [27]:
#Create a Payment Summary Code
payment_summary = (
    payments
    .groupby("order_id")
    .agg(
        total_payment=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
        payment_type=("payment_type", lambda x: ", ".join(sorted(x.unique()))),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

In [31]:
print(payment_summary.shape)
payment_summary.head()

(99440, 5)


,order_id,total_payment,payment_count,payment_type,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,credit_card,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,credit_card,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,credit_card,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,credit_card,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,credit_card,3


In [32]:
fact_orders = pd.merge(
    fact_orders,
    payment_summary,
    on="order_id",
    how="left"
)

In [36]:
fact_orders.shape

(99441, 19)

In [37]:

fact_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_after_carrier_flag,late_delivery_flag,delivery_before_carrier_flag,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment,payment_count,payment_type,max_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False,False,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,3.00,"credit_card, voucher",1.00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False,False,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.00,boleto,1.00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False,False,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,1.00,credit_card,3.00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False,False,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1.00,credit_card,1.00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False,False,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1.00,credit_card,1.00


In [38]:
fact_orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
approval_after_carrier_flag                bool
late_delivery_flag                         bool
delivery_before_carrier_flag               bool
customer_unique_id                          str
customer_zip_code_prefix                  int64
customer_city                               str
customer_state                              str
total_payment                           float64
payment_count                           float64
payment_type                                str
max_installments                        float64
dtype: object

## check reviews assumption 

In [39]:
reviews.groupby("order_id").size().value_counts().sort_index()

1    98126
2      543
3        4
Name: count, dtype: int64

### we have again 1:M many relationship if we merge reviews with orders it can duplicate reviews 

In [41]:
# step 1SORT TO GET THE MOST RECENT REVIEW FOR EACH ORDER
reviews_sorted = reviews.sort_values(
    by="review_answer_timestamp"
)

reviews_sorted.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
37547,6916ca4502d6d3bfd39818759d55d536,bfbd0f9bdef84302105ad712db648a6c,1,NaN,nao recebi o produto e nem resposta da empresa,2016-10-06 00:00:00,2016-10-07 18:32:28
5503,49f695dffa457eaba90d388a5c37e942,e5215415bb6f76fe3b7cb68103a0d1c0,1,NaN,"PRODUTO NÃO CHEGOU,E JÁ PASSOU O PRAZO DE ENTREGA",2016-10-09 00:00:00,2016-10-11 14:31:29
60439,743d98b1a4782f0646898fc915ef002a,e2144124f98f3bf46939bc5183104041,4,NaN,NaN,2016-10-15 00:00:00,2016-10-16 03:20:17
28075,53752edb26544dd41c1209f582c9c589,b8b9d7046c083150cb5360b83a8ebb51,5,NaN,O pedido foi entregue antes do prazo pr0metido,2016-10-16 01:00:00,2016-10-16 15:45:11
41042,b2d5d8db2a841d27a72e4c06c6212368,9aa3197e4887919fde0307fc23601d7a,4,NaN,Só chegou uma parte do pedido ate agora..,2016-10-15 00:00:00,2016-10-17 21:02:49


#### Step 2 – Keep the Latest Review

In [42]:
reviews_summary = (
    reviews_sorted
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
)

In [43]:
reviews_summary.shape

(98673, 7)

#### Step 4 – Merge

In [44]:
fact_orders = pd.merge(
    fact_orders,
    reviews_summary[
        [
            "order_id",
            "review_score",
            "review_comment_title",
            "review_comment_message",
        ]
    ],
    on="order_id",
    how="left"
)

In [45]:
fact_orders.shape

(99441, 22)

In [70]:
fact_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,approval_after_carrier_flag,late_delivery_flag,delivery_before_carrier_flag,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment,payment_count,payment_type,max_installments,review_score,review_comment_title,review_comment_message
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False,False,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,3.00,"credit_card, voucher",1.00,4.00,NaN,"Não testei o produto ainda, mas ele veio corre..."
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False,False,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.00,boleto,1.00,4.00,Muito boa a loja,Muito bom o produto.
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False,False,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,1.00,credit_card,3.00,5.00,NaN,NaN
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False,False,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1.00,credit_card,1.00,5.00,NaN,O produto foi exatamente o que eu esperava e e...
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False,False,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1.00,credit_card,1.00,5.00,NaN,NaN


#### payments are merged.

In [85]:
print(fact_orders.shape)

print(fact_orders["order_id"].nunique())

(99441, 22)
99441


## NEW TABLE SALES-FACT TO FIND AND LIST SALES KPI

In [96]:
sales_fact = order_items_etl.copy()
sales_fact.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [97]:

# Merge Products
sales_fact = pd.merge(
    sales_fact,
    products,
    on="product_id",
    how="left"
)

# Merge Sellers
sales_fact = pd.merge(
    sales_fact,
    sellers,
    on="seller_id",
    how="left"
)

# Merge selected columns from fact_orders
order_columns = [
    "order_id",
    "customer_id",
    "order_purchase_timestamp",
    "order_status",
    "customer_state",
    "customer_city",
    "late_delivery_flag",
    "review_score"
]

sales_fact = pd.merge(
    sales_fact,
    fact_orders[order_columns],
    on="order_id",
    how="left"
)

In [100]:
fact_orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'approval_after_carrier_flag', 'late_delivery_flag',
       'delivery_before_carrier_flag', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'total_payment', 'payment_count', 'payment_type', 'max_installments',
       'review_score', 'review_comment_title', 'review_comment_message'],
      dtype='str')

In [98]:
sales_fact.columns


Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'seller_zip_code_prefix', 'seller_city', 'seller_state', 'customer_id',
       'order_purchase_timestamp', 'order_status', 'customer_state',
       'customer_city', 'late_delivery_flag', 'review_score'],
      dtype='str')

In [99]:
import os

os.makedirs("processed", exist_ok=True)

fact_orders.to_csv("C:\\Users\\Steve\\Desktop\\datascience-project1\\project\\data\\processed/fact_orders.csv", index=False)
sales_fact.to_csv("C:\\Users\\Steve\\Desktop\\datascience-project1\\project\\data\\processed/sales_fact.csv", index=False)